# Task 3 — Segmentation (§2.3 Transfer Learning · §2.6 Evaluation · §2.7 Pipeline · §2.8 Meta)

**Colab setup:** Runtime → Change runtime type → **T4 GPU**

**Prerequisites:** Task 1 done — `classifier.pth` must exist in your Drive.

**After this notebook you will have:**
- `unet.pth` saved directly to your Google Drive
- W&B runs for all 3 freeze modes (§2.3) + sample masks (§2.6) + pipeline table (§2.7)
- `UNET_DRIVE_ID` updated in `models/multitask.py` and pushed to GitHub
- Ready to submit **4.3a / 4.3b / 4.3c** to Gradescope

## 1 · Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
CKPT_DIR = "/content/drive/MyDrive/da6401_checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

cls_path = f"{CKPT_DIR}/classifier.pth"
if os.path.exists(cls_path):
    print(f"✅  classifier.pth found ({os.path.getsize(cls_path)/1e6:.1f} MB) — ready to proceed")
else:
    print("❌  classifier.pth NOT found — run Task 1 notebook first!")

## 2 · Setup

In [ ]:
import os

os.environ["WANDB_API_KEY"] = "PASTE_YOUR_WANDB_API_KEY_HERE"

GITHUB_TOKEN    = "PASTE_YOUR_GITHUB_TOKEN_HERE"
GITHUB_USERNAME = "usnaveen"
GITHUB_REPO     = "A2_Deep_Learning"

!git clone https://{GITHUB_TOKEN}@github.com/{GITHUB_USERNAME}/{GITHUB_REPO}.git da6401_assignment_2
%cd da6401_assignment_2

!pip install -q wandb albumentations scikit-learn

import torch
print(f"CUDA available: {torch.cuda.is_available()}")

## 3 · Download Dataset

In [ ]:
from data.pets_dataset import download_oxford_pet
download_oxford_pet("./data/oxford_pet")

## 4 · §2.3 — Transfer Learning Showdown
Three runs: frozen / partial / full fine-tuning. All logged to W&B for comparison.

In [ ]:
CKPT_DIR = "/content/drive/MyDrive/da6401_checkpoints"
!python train.py --task segment --experiment exp-transfer --freeze-mode frozen  --epochs 60 --lr 1e-3  --batch-size 16 --num-workers 2 --checkpoint-dir {CKPT_DIR}

In [ ]:
CKPT_DIR = "/content/drive/MyDrive/da6401_checkpoints"
!python train.py --task segment --experiment exp-transfer --freeze-mode partial --epochs 60 --lr 5e-4  --batch-size 16 --num-workers 2 --checkpoint-dir {CKPT_DIR}

In [ ]:
CKPT_DIR = "/content/drive/MyDrive/da6401_checkpoints"
# Full fine-tuning — this is the best model and gets saved as unet.pth
!python train.py --task segment --experiment exp-transfer --freeze-mode full    --epochs 80 --lr 3e-4  --batch-size 16 --num-workers 2 --checkpoint-dir {CKPT_DIR}

## 5 · Verify Dice Score

In [ ]:
import torch, numpy as np
from data.pets_dataset import create_dataloaders
from models.segmentation import VGG11UNet

CKPT_DIR = "/content/drive/MyDrive/da6401_checkpoints"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
_, _, test_loader = create_dataloaders(root="./data/oxford_pet", batch_size=16, num_workers=2)

model = VGG11UNet(num_classes=3).to(device)
model.load_state_dict(torch.load(f"{CKPT_DIR}/unet.pth", map_location=device, weights_only=False))
model.eval()

def dice(pred, target, nc=3, eps=1e-6):
    pf, tf = pred.view(-1), target.view(-1)
    return np.mean([(2*((pf==c).float()*(tf==c).float()).sum().item()+eps) /
                    ((pf==c).float().sum().item()+(tf==c).float().sum().item()+eps) for c in range(nc)])

total_dice, n = 0.0, 0
with torch.no_grad():
    for batch in test_loader:
        preds = model(batch["image"].to(device)).argmax(1)
        masks = batch["mask"].to(device)
        total_dice += dice(preds, masks) * len(preds)
        n += len(preds)

print(f"\n✅  Test Macro-Dice = {total_dice/n:.4f}")
print("   need > 0.30 for 4.3a · > 0.50 for 4.3b · > 0.80 for 4.3c")

## 6 · §2.7 — Final Pipeline Showcase
Logs a W&B table showing classification + localization + segmentation on the same images.

In [ ]:
import wandb, torch, numpy as np
import matplotlib.pyplot as plt, matplotlib.patches as patches
from io import BytesIO
from data.pets_dataset import create_dataloaders, IMAGENET_MEAN, IMAGENET_STD, OxfordIIITPetDataset
from models.multitask import MultiTaskPerceptionModel

CKPT_DIR = "/content/drive/MyDrive/da6401_checkpoints"
device   = torch.device("cuda" if torch.cuda.is_available() else "cpu")
_, _, test_loader = create_dataloaders(root="./data/oxford_pet", batch_size=8, num_workers=2)

model = MultiTaskPerceptionModel(
    classifier_path=f"{CKPT_DIR}/classifier.pth",
    localizer_path =f"{CKPT_DIR}/localizer.pth",
    unet_path      =f"{CKPT_DIR}/unet.pth",
).to(device)
model.eval()

COLORMAP = np.array([[0,200,0],[40,40,40],[255,255,0]], dtype=np.uint8)
CLASS_NAMES = OxfordIIITPetDataset.CLASS_NAMES

def denorm(t):
    m=torch.tensor(IMAGENET_MEAN).view(3,1,1); s=torch.tensor(IMAGENET_STD).view(3,1,1)
    return ((t.cpu()*s+m).clamp(0,1).permute(1,2,0).numpy()*255).astype(np.uint8)

run   = wandb.init(project="da6401-assignment2", name="pipeline_showcase", tags=["exp-pipeline"])
table = wandb.Table(columns=["Image+BBox", "GT Class", "Pred Class", "IoU", "GT Mask", "Pred Mask"])

count = 0
with torch.no_grad():
    for batch in test_loader:
        out  = model(batch["image"].to(device))
        pred_cls   = out["classification"].cpu().argmax(1)
        pred_boxes = out["localization"].cpu()
        pred_masks = out["segmentation"].cpu().argmax(1)

        for i in range(batch["image"].size(0)):
            if count >= 20: break
            gt  = batch["bbox"][i].numpy()
            pred= pred_boxes[i].numpy()
            px1,py1,px2,py2 = pred[0]-pred[2]/2,pred[1]-pred[3]/2,pred[0]+pred[2]/2,pred[1]+pred[3]/2
            tx1,ty1,tx2,ty2 = gt[0]-gt[2]/2,gt[1]-gt[3]/2,gt[0]+gt[2]/2,gt[1]+gt[3]/2
            inter = max(0,min(px2,tx2)-max(px1,tx1))*max(0,min(py2,ty2)-max(py1,ty1))
            iou   = inter/((px2-px1)*(py2-py1)+(tx2-tx1)*(ty2-ty1)-inter+1e-6)

            fig, ax = plt.subplots(figsize=(4,4))
            ax.imshow(denorm(batch["image"][i]))
            ax.add_patch(patches.Rectangle((gt[0]-gt[2]/2,gt[1]-gt[3]/2),gt[2],gt[3],lw=2,ec='green',fc='none',label='GT'))
            ax.add_patch(patches.Rectangle((pred[0]-pred[2]/2,pred[1]-pred[3]/2),pred[2],pred[3],lw=2,ec='red',fc='none',label='Pred'))
            ax.legend(fontsize=7); ax.axis("off"); plt.tight_layout()
            buf = BytesIO(); fig.savefig(buf,format='png',dpi=80,bbox_inches='tight'); buf.seek(0); plt.close(fig)

            table.add_data(
                wandb.Image(buf),
                CLASS_NAMES[batch["label"][i].item()],
                CLASS_NAMES[pred_cls[i].item()],
                round(iou,3),
                wandb.Image(COLORMAP[batch["mask"][i].numpy()]),
                wandb.Image(COLORMAP[pred_masks[i].numpy()]),
            )
            count += 1
        if count >= 20: break

wandb.log({"pipeline_showcase": table})
wandb.finish()
print("✅  §2.7 pipeline table logged to W&B")

## 7 · Get Drive File ID + Make Public

In [ ]:
from google.colab import auth
from googleapiclient.discovery import build

auth.authenticate_user()
drive_svc = build('drive', 'v3')

results = drive_svc.files().list(
    q="name='unet.pth' and trashed=false",
    fields="files(id, name)"
).execute()

files = results.get('files', [])
if not files:
    print("❌  unet.pth not found in Drive — did training finish?")
else:
    UNET_DRIVE_ID = files[0]['id']
    drive_svc.permissions().create(
        fileId=UNET_DRIVE_ID,
        body={'type': 'anyone', 'role': 'reader'}
    ).execute()
    print(f"✅  UNET_DRIVE_ID = {UNET_DRIVE_ID}")

## 8 · Update `multitask.py` + Push to GitHub

In [ ]:
import re

multitask_path = "models/multitask.py"
with open(multitask_path) as f:
    content = f.read()

content = re.sub(
    r'UNET_DRIVE_ID\s*=\s*"[^"]*"',
    f'UNET_DRIVE_ID       = "{UNET_DRIVE_ID}"',
    content
)

with open(multitask_path, "w") as f:
    f.write(content)

!git config user.email "naveen@student.iitm.ac.in"
!git config user.name  "Naveen"
!git add models/multitask.py
!git commit -m "task3: set UNET_DRIVE_ID for Gradescope"
!git push

print("\n🎉  All done! Submit to Gradescope to check 4.3a / 4.3b / 4.3c")